# SI7011 — Taller 4 TweetEval
Daniel Restrepo — CPU


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']=''
!pip install -q transformers accelerate peft datasets evaluate scikit-learn matplotlib seaborn
import torch; print('CPU', torch.__version__)


## tweeteval-part-1-data


# Parte 1 — Datos y Tokenización
### Workshop: Clasificación de Emociones en Twitter

Este notebook cubre:
- Exploración del dataset TweetEval Emotion (EDA)
- Análisis comparativo de tokenización: DistilBERT vs BERTweet
- Guarda el dataset crudo para que los siguientes notebooks lo carguen directamente

> **Referencia:** Barbieri et al. (2020). *TweetEval: Unified Benchmark and Comparative Evaluation for Tweet Classification*. EMNLP Findings.

In [ ]:
# !pip install 'transformers[torch]' 'accelerate>=1.1.0' datasets evaluate scikit-learn matplotlib seaborn -q

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

SEED        = 42
MAX_LENGTH  = 128
LABEL_NAMES = ["anger", "joy", "optimism", "sadness"]
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
PALETTE     = {"anger": "#e74c3c", "joy": "#f1c40f",
               "optimism": "#2ecc71", "sadness": "#3498db"}

np.random.seed(SEED)
torch.manual_seed(SEED)
print("Config OK")

## 1. Carga del dataset

In [ ]:
raw = load_dataset("tweet_eval", "emotion")
print(raw)
print("\nEjemplos por split:")
for split in raw:
    print(f"  {split}: {len(raw[split])} ejemplos")

In [ ]:
print("Primeros 6 ejemplos del split de entrenamiento:\n")
for ex in raw["train"].select(range(6)):
    print(f"  [{ID2LABEL[ex['label']].upper():<10}]  {ex['text']}")

## 2. EDA

In [ ]:
df = raw["train"].to_pandas()
df["emotion"]    = df["label"].map(ID2LABEL)
df["n_chars"]    = df["text"].str.len()
df["n_words"]    = df["text"].str.split().str.len()
df["n_hashtags"] = df["text"].str.count(r"#\w+")
df["n_mentions"] = df["text"].str.count(r"@\w+")

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
counts = df["emotion"].value_counts().reindex(LABEL_NAMES)
bars = ax1.bar(counts.index, counts.values,
               color=[PALETTE[k] for k in counts.index], edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val}", ha="center", fontweight="bold", fontsize=9)
ax1.set_title("Distribución de clases (train)", fontweight="bold")
ax1.set_ylabel("Ejemplos")
ax1.spines[["top", "right"]].set_visible(False)

ax2 = fig.add_subplot(gs[0, 1:3])
for emotion in LABEL_NAMES:
    subset = df[df["emotion"] == emotion]
    ax2.hist(subset["n_words"], bins=30, alpha=0.55,
             label=emotion, color=PALETTE[emotion], edgecolor="none")
ax2.set_title("Distribución de longitud por emoción (palabras)", fontweight="bold")
ax2.set_xlabel("Palabras"); ax2.set_ylabel("Frecuencia")
ax2.legend(); ax2.spines[["top", "right"]].set_visible(False)

ax3 = fig.add_subplot(gs[1, 0])
means = df.groupby("emotion")[["n_hashtags", "n_mentions"]].mean().reindex(LABEL_NAMES)
x = np.arange(len(LABEL_NAMES)); w = 0.35
ax3.bar(x - w/2, means["n_hashtags"], w, label="#hashtags", color="#8e44ad", edgecolor="white")
ax3.bar(x + w/2, means["n_mentions"], w, label="@mentions", color="#16a085", edgecolor="white")
ax3.set_xticks(x); ax3.set_xticklabels(LABEL_NAMES)
ax3.set_title("Media de #hashtags y @mentions", fontweight="bold")
ax3.legend(); ax3.spines[["top", "right"]].set_visible(False)

ax4 = fig.add_subplot(gs[1, 1:3])
df.boxplot(column="n_words", by="emotion",
           positions=range(len(LABEL_NAMES)), ax=ax4,
           patch_artist=True, boxprops=dict(facecolor="#ecf0f1"),
           medianprops=dict(color="red", linewidth=2))
ax4.set_xticklabels(LABEL_NAMES)
ax4.set_title("Longitud por emoción", fontweight="bold")
ax4.set_xlabel(""); ax4.set_ylabel("Palabras")
plt.sca(ax4); plt.title("")

fig.suptitle("TweetEval Emotion — EDA (split train)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print("\nConteo de clases en cada split:")
for split in raw:
    c = Counter(raw[split]["label"])
    total = sum(c.values())
    print(f"  {split}: " + "  ".join(
        f"{ID2LABEL[k]}={v} ({v/total*100:.0f}%)" for k, v in sorted(c.items())))

### Observaciones del EDA

Antes de continuar, anota tus observaciones:

- ¿Hay desequilibrio de clases? ¿Cuál es la clase mayoritaria?
- ¿Cuántos tokens tienen aproximadamente los tweets? ¿Justifica `MAX_LENGTH=128`?
- ¿Qué características de Twitter (hashtags, mentions) podrían afectar a un tokenizador entrenado en texto formal?

> ⚠️ Con clases desbalanceadas la *accuracy* puede ser engañosa. Usaremos **F1 macro** como métrica principal.

## 3. Tokenización comparativa

DistilBERT fue pre-entrenado en Wikipedia y BookCorpus — texto formal. Su tokenizador no conoce abreviaciones de Twitter, hashtags ni emojis.

BERTweet fue pre-entrenado en 850M tweets. Su tokenizador fue entrenado sobre texto de Twitter, incluyendo `@mentions`, `#hashtags` y emojis normalizados.

In [ ]:
tok_distilbert = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tok_bertweet   = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True)

def compare_tokenization(text, label=""):
    toks_d = tok_distilbert.tokenize(text)
    toks_b = tok_bertweet.tokenize(text)
    print(f"\n{'─'*65}")
    print(f"  [{label}] {text}" if label else f"  {text}")
    print(f"{'─'*65}")
    print(f"  DistilBERT  ({len(toks_d):2d} tokens): {toks_d}")
    print(f"  BERTweet    ({len(toks_b):2d} tokens): {toks_b}")

examples = [
    ("I'm so happy today!!! 😊",                        "joy"),
    ("this is absolutely disgusting #angry #wtf",       "anger"),
    ("@user can't believe what just happened... 😢",    "sadness"),
    ("RT @user: The future looks bright #hope #goals",  "optimism"),
]

for text, label in examples:
    compare_tokenization(text, label)

### 📝 TODO 2.1 — Análisis de tokenización

1. ¿Cuántos tokens produce en promedio cada tokenizador sobre todos los tweets del split de train?
2. ¿Cómo tokeniza DistilBERT el hashtag `#wtf` comparado con BERTweet?
3. ¿Qué pasa con `@user`? ¿Y con emojis como `😊`?
4. Busca 2 tweets del dataset (`raw["train"][i]["text"]`) que creas que mostrarán diferencias interesantes y analízalos.

In [ ]:
# TODO 2.1 — Análisis de tokenización
train_texts = raw["train"]["text"]

def token_lengths(tokenizer, texts):
    return [len(tokenizer(t, truncation=False)["input_ids"]) for t in texts]

len_d = token_lengths(tok_distilbert, train_texts)
len_b = token_lengths(tok_bertweet, train_texts)
print(f"Promedio tokens (train) — DistilBERT: {np.mean(len_d):.2f} | BERTweet: {np.mean(len_b):.2f}")
print(f"BERTweet produce secuencias {'más largas' if np.mean(len_b) > np.mean(len_d) else 'más cortas'} en promedio.")

compare_tokenization("this is absolutely disgusting #angry #wtf", "anger")
compare_tokenization("@user can't believe what just happened... 😢", "sadness")

for ex in raw["train"]:
    if "#" in ex["text"] or "@" in ex["text"] or "😊" in ex["text"]:
        compare_tokenization(ex["text"], ID2LABEL[ex["label"]])
        break
for ex in raw["train"]:
    if len(ex["text"]) > 120:
        compare_tokenization(ex["text"][:120] + "...", ID2LABEL[ex["label"]])
        break

def pct_over_max(lengths, max_length=128):
    return 100 * sum(l > max_length for l in lengths) / len(lengths)

print(f"% tweets > {MAX_LENGTH} tokens — DistilBERT: {pct_over_max(len_d):.2f}% | BERTweet: {pct_over_max(len_b):.2f}%")


<details>
<summary>💡 Pista</summary>

Para el punto 1, itera sobre `raw["train"]["text"]` y llama `tokenizer(text)["input_ids"]` en cada tweet. Recuerda que `len(input_ids)` incluye los tokens especiales `[CLS]` y `[SEP]`.

</details>

## tweeteval-part-2-pipeline


# Parte 2 — Pipeline Reutilizable
### Workshop: Clasificación de Emociones en Twitter

Este notebook define las funciones compartidas que usan todos los experimentos:
- Métricas (`compute_metrics`)
- Tokenización (`make_tokenized_dataset`)
- Evaluación completa (`full_evaluation`)
- Curvas de entrenamiento (`plot_training_curves`)
- Constructor de Trainer (`make_trainer`)

**Prerequisito:** haber ejecutado `part-1-data.ipynb`

> En lugar de repetir código en cada notebook, definimos todo aquí. En la práctica esto se convertiría en un módulo Python importable.

In [ ]:
# !pip install 'transformers[torch]' 'accelerate>=1.1.0' peft datasets evaluate scikit-learn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import evaluate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED        = 42
MAX_LENGTH  = 128
BATCH_SIZE  = 32
EPOCHS      = 1
LABEL_NAMES = ["anger", "joy", "optimism", "sadness"]
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS  = len(LABEL_NAMES)

np.random.seed(SEED)
torch.manual_seed(SEED)

# Cargar el dataset una vez — todos los experimentos lo usan
raw = load_dataset("tweet_eval", "emotion")
print(raw)

## Métricas

In [ ]:
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")
    acc = acc_metric.compute(predictions=preds, references=labels)
    return {"f1_macro": f1["f1"], "accuracy": acc["accuracy"]}

print("compute_metrics OK")

## Tokenización

In [ ]:
def make_tokenized_dataset(tokenizer, max_length=MAX_LENGTH):
    """Tokeniza los tres splits con el tokenizador dado."""
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    return raw.map(tokenize, batched=True, remove_columns=["text"])

print("make_tokenized_dataset OK")

## Evaluación completa

In [ ]:
def full_evaluation(trainer, test_dataset, model_name=""):
    """Classification report + confusion matrix para el split de test."""
    output = trainer.predict(test_dataset)
    preds  = np.argmax(output.predictions, axis=1)
    labels = output.label_ids

    title = f"Test — {model_name}" if model_name else "Test"
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(classification_report(labels, preds,
                                 target_names=LABEL_NAMES, digits=4))

    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(
        ax=axes[0], colorbar=False, cmap="Blues", xticks_rotation=30)
    axes[0].set_title("Confusion Matrix (conteos)", fontweight="bold")
    ConfusionMatrixDisplay(cm_norm, display_labels=LABEL_NAMES).plot(
        ax=axes[1], colorbar=False, cmap="Blues",
        values_format=".2f", xticks_rotation=30)
    axes[1].set_title("Confusion Matrix (normalizada)", fontweight="bold")
    plt.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

    return output.metrics

print("full_evaluation OK")

## Curvas de entrenamiento

In [ ]:
def plot_training_curves(trainer, title=""):
    logs       = pd.DataFrame(trainer.state.log_history)
    train_logs = logs[logs["loss"].notna()][["step", "loss", "learning_rate"]]
    eval_logs  = logs[logs["eval_loss"].notna()][
        ["epoch", "eval_loss", "eval_f1_macro", "eval_accuracy"]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(train_logs["step"], train_logs["loss"],
                 color="steelblue", linewidth=1.5)
    axes[0].set_title("Training Loss", fontweight="bold")
    axes[0].set_xlabel("Paso"); axes[0].set_ylabel("Loss")
    axes[0].spines[["top", "right"]].set_visible(False)

    axes[1].plot(eval_logs["epoch"], eval_logs["eval_f1_macro"],
                 marker="o", color="#e74c3c", linewidth=2, label="F1 macro")
    axes[1].plot(eval_logs["epoch"], eval_logs["eval_accuracy"],
                 marker="s", color="#2ecc71", linewidth=2, label="Accuracy")
    axes[1].set_title("Métricas de Validación", fontweight="bold")
    axes[1].set_xlabel("Época"); axes[1].set_ylabel("Score")
    axes[1].set_ylim(0.3, 1.0); axes[1].legend()
    axes[1].spines[["top", "right"]].set_visible(False)

    axes[2].plot(train_logs["step"], train_logs["learning_rate"],
                 color="purple", linewidth=1.5)
    axes[2].set_title("Learning Rate Schedule", fontweight="bold")
    axes[2].set_xlabel("Paso"); axes[2].set_ylabel("LR")
    axes[2].spines[["top", "right"]].set_visible(False)

    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

print("plot_training_curves OK")

## Constructor de Trainer

In [ ]:
def make_trainer(model, tokenizer, tokenized_ds, output_dir,
                 lr=2e-5, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Construye un Trainer con configuración estándar."""
    n_train_steps = (len(tokenized_ds["train"]) // batch_size) * epochs

    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = lr,
        weight_decay                = 0.01,
        warmup_steps                = int(0.1 * n_train_steps),
        lr_scheduler_type           = "linear",
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1_macro",
        greater_is_better           = True,
        logging_steps               = 20,
        report_to                   = "none",
        seed                        = SEED,
        fp16                        = False,  # evita CUDA kernel errors en algunos entornos Kaggle
    )

    return Trainer(
        model             = model,
        args              = args,
        train_dataset     = tokenized_ds["train"],
        eval_dataset      = tokenized_ds["validation"],
        processing_class  = tokenizer,
        data_collator     = DataCollatorWithPadding(tokenizer),
        compute_metrics   = compute_metrics,
        callbacks         = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

print("make_trainer OK")
print("\nPipeline lista. Puedes continuar con part-3-distilbert.ipynb")

## tweeteval-part-3-distilbert


# Parte 3 — Experimento A: DistilBERT
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `distilbert-base-uncased`  
**Pre-entrenamiento:** Wikipedia + BookCorpus (texto formal)  
**Tokenizador:** WordPiece

Este es nuestro **baseline**. Pre-entrenado en texto formal, sin conocimiento específico del lenguaje de Twitter.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
HF_REPO          = "your-username/tweeteval-emotion-distilbert"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 3.1 — Tokenizar el dataset con DistilBERT

Usa `make_tokenized_dataset` con el tokenizador correcto y guarda el resultado en `ds_distilbert`.

In [ ]:
tok_distilbert = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
ds_distilbert  = make_tokenized_dataset(tok_distilbert)
print(ds_distilbert)


### 📝 TODO 3.2 — Cargar el modelo y contar parámetros

In [ ]:
from transformers import AutoModelForSequenceClassification
model_distilbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
total = sum(p.numel() for p in model_distilbert.parameters())
trainable = sum(p.numel() for p in model_distilbert.parameters() if p.requires_grad)
print(f"Parámetros totales: {total/1e6:.1f}M | entrenables: {trainable/1e6:.1f}M")


### 📝 TODO 3.3 — Entrenar y evaluar

In [ ]:
trainer_distilbert = make_trainer(
    model_distilbert, tok_distilbert, ds_distilbert,
    output_dir="./checkpoints/distilbert", lr=LR,
)
trainer_distilbert.train()
plot_training_curves(trainer_distilbert, title="DistilBERT")
metrics_distilbert = full_evaluation(trainer_distilbert, ds_distilbert["test"], model_name="DistilBERT")
metrics_distilbert


## Push to Hub

## tweeteval-part-4-bertweet


# Parte 4 — Experimento B: BERTweet
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `vinai/bertweet-base`  
**Pre-entrenamiento:** 850M tweets en inglés  
**Tokenizador:** BPE entrenado sobre texto de Twitter

BERTweet tiene la misma arquitectura que BERT-base (12 capas, 768 dimensiones, ~110M parámetros), pero fue pre-entrenado desde cero sobre tweets.

> **Nota sobre `normalization=True`:** el tokenizador normaliza las URLs a `HTTPURL` y los @mentions a `@USER` antes de tokenizar, replicando el preprocesamiento usado durante el pre-entrenamiento.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 4.1 — Tokenizar el dataset con BERTweet

Importante: carga el tokenizador con `normalization=True`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)


### 📝 TODO 4.2 — Cargar el modelo BERTweet

Mismo procedimiento que con DistilBERT pero con `MODEL_CHECKPOINT`. Imprime el número de parámetros y compáralo con DistilBERT.

In [ ]:
model_bertweet = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
total = sum(p.numel() for p in model_bertweet.parameters())
trainable = sum(p.numel() for p in model_bertweet.parameters() if p.requires_grad)
print(f"Parámetros totales: {total/1e6:.1f}M | entrenables: {trainable/1e6:.1f}M")


### 📝 TODO 4.3 — Entrenar y evaluar BERTweet

Repite el mismo proceso del TODO 3.3 con el modelo y dataset de BERTweet.

**Pregunta:** ¿necesitarías cambiar el learning rate para BERTweet vs DistilBERT? ¿Por qué sí o por qué no?

In [ ]:
trainer_bertweet = make_trainer(
    model_bertweet, tok_bertweet, ds_bertweet,
    output_dir="./checkpoints/bertweet", lr=LR,
)
trainer_bertweet.train()
plot_training_curves(trainer_bertweet, title="BERTweet")
metrics_bertweet = full_evaluation(trainer_bertweet, ds_bertweet["test"], model_name="BERTweet")
metrics_bertweet


## Push to Hub

## tweeteval-part-5-lora


# Parte 5 — Bonus: LoRA sobre BERTweet
### Workshop: Clasificación de Emociones en Twitter

BERTweet tiene ~110M parámetros — el doble que DistilBERT. LoRA permite adaptarlo con una fracción mínima de los parámetros.

BERTweet usa la arquitectura de BERT, cuyas proyecciones de atención tienen nombres distintos a DistilBERT:

| Modelo | Proyecciones |
|---|---|
| DistilBERT | `q_lin`, `k_lin`, `v_lin`, `out_lin` |
| BERTweet / BERT | `query`, `key`, `value`, `dense` |

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet-lora"  # <-- cambia esto
LR_LORA          = 2e-4   # más alto que full FT — pocos parámetros entrenables

## Dataset tokenizado

Reutilizamos el tokenizador de BERTweet del experimento anterior.

In [ ]:
from transformers import AutoTokenizer

tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)

## Inspección de módulos

Antes de aplicar LoRA, inspeccionamos los nombres de las capas lineales de BERTweet.

In [ ]:
from transformers import AutoModelForSequenceClassification

base_bt = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID)

print("Módulos lineales de BERTweet:")
for name, module in base_bt.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"  {name}  [{module.in_features} → {module.out_features}]")

### 📝 TODO 5.1 — Configurar y aplicar LoRA a BERTweet

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
)
model_bertweet_lora = get_peft_model(base_bt, lora_config)
model_bertweet_lora.print_trainable_parameters()


### 📝 TODO 5.2 — Entrenar BERTweet-LoRA

In [ ]:
trainer_lora = make_trainer(
    model_bertweet_lora, tok_bertweet, ds_bertweet,
    output_dir="./checkpoints/bertweet-lora", lr=LR_LORA,
)
trainer_lora.train()
plot_training_curves(trainer_lora, title="BERTweet-LoRA")
metrics_bertweet_lora = full_evaluation(trainer_lora, ds_bertweet["test"], model_name="BERTweet-LoRA")
metrics_bertweet_lora
